# 365 Probabilidades · Dia #064
## Qual a probabilidade de ter um porquê mudar a sua data de morte?

**Tipo:** Preditivo
**Data de publicação:** 2026-08-16
**Ferramenta:** Python
**Decisão analisada:** Ter um porquê muda alguma coisa fora da minha cabeça?
**Hashtag:** #365Probabilidades #Dia064

---

### 📖 A História

Propósito é uma das palavras mais gastas do vocabulário corporativo. Aparece em
crachá, em parede de escritório, em slide de treinamento motivacional. De tanto
ser usada para vender coisa, virou palavra sem peso.

O problema de palavras assim é que fica difícil perguntar se elas significam
alguma coisa mensurável. Quem pergunta parece ingênuo, e quem responde
normalmente está vendendo algo.

Mas dá para fazer a pergunta de um jeito que não depende de opinião nenhuma.

Você mede o senso de propósito de dezenas de milhares de pessoas hoje. Espera.
E depois vai conferir, em registro de óbito, quem morreu e quando.

O desfecho não é autorrelatado. Ninguém responde questionário sobre a própria
morte. É o tipo raro de pergunta subjetiva com resposta objetiva do outro lado.

Foi exatamente isso que a literatura fez, e o resultado saiu maior do que eu
esperava.

---

### 📚 O Conceito: o que um hazard ratio quer dizer

Hazard ratio é a razão entre a taxa instantânea de um evento em dois grupos ao
longo do tempo. Um HR de 0,76 quer dizer que, a cada instante do seguimento, o
grupo com mais propósito morre a uma taxa 24% menor que o grupo de comparação.

Não é probabilidade de morrer. É velocidade de morrer. Todo mundo morre no fim,
e o que a estatística de sobrevivência mede é o ritmo.

Duas advertências que valem para o dia inteiro.

A primeira: **isso é coorte observacional, portanto associação, não causa.**
Ninguém sorteou propósito para metade da amostra.

A segunda: **causalidade reversa é plausível.** Quem já está doente tende a
relatar menos propósito, e também tende a morrer antes. Parte da associação pode
ser a doença aparecendo nas duas pontas. Os autores atacam isso com ajuste por
fatores clínicos, e é justamente aí que o número encolhe.

---

### 🧮 O Modelo

O dia tem uma coorte pequena e famosa, uma meta-análise gigante e recente, e a
distância entre as duas.

**Fontes:**
- Sutin, A. R., Luchetti, M., Stephan, Y., Terracciano, A. e colegas, 2026 ·
  *Psychological Medicine* · meta-análise de dados individuais (k=8) combinada com
  revisão sistemática (k=17 amostras de 14 publicações) · **25 amostras,
  N=488.765, 48.928 óbitos, até 32 anos de seguimento** ·
  **HR = 0,76 [IC 95%: 0,70 · 0,83]** · ajustado por fatores comportamentais e
  clínicos **HR = 0,85 [0,82 · 0,89]** · ajustado por depressão
  **HR = 0,91 [0,88 · 0,94]**
- Alimujiang, A., Wiensch, A., Boss, J., Fleischer, N. L., Mondul, A. M.,
  McLean, K., Mukherjee, B. & Pearce, C. L., 2019 · *JAMA Network Open* 2(5),
  e194270 · Health and Retirement Study · **N=6.985**, 57,5% mulheres, idade média
  68,6 anos (DP 9,8) · **HR = 2,43 [IC 95%: 1,57 · 3,75]** comparando a categoria
  mais baixa de propósito com a mais alta · causas cardíacas, circulatórias e
  sanguíneas **HR = 2,66 [1,62 · 4,38]**
- Cohen, R., Bavishi, C. & Rozanski, A., 2016 · *Psychosomatic Medicine* 78(2),
  122-133 · **10 estudos prospectivos, N=136.265** · registro histórico de que a
  pergunta já vinha sendo medida, sem número de capa neste dia

**Precisão obrigatória sobre os contrastes:** o HR de 2,43 do Alimujiang compara
**extremos de categoria** (menor propósito contra maior propósito). O HR
meta-analítico de 0,76 é **por unidade da escala**. Os dois números medem a mesma
direção, mas não são intercambiáveis e não devem ser comparados ponto a ponto.

**Nota metodológica sobre o fator ×0.80:** não se aplica. Coorte prospectiva com
desfecho objetivo. O propósito é autorrelatado, mas o óbito vem de registro, e é
o óbito que está sendo previsto.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

print("Bibliotecas carregadas")

In [ ]:
# --- DADOS DA LITERATURA ---

# Sutin, Luchetti, Stephan, Terracciano e colegas, 2026, Psychological Medicine
# 25 amostras (8 de dados individuais + 17 publicadas), 3 continentes
n_meta        = 488_765
obitos_meta   = 48_928
seguimento    = 32          # anos, maximo
k_amostras    = 25

# HR por unidade da escala de proposito, em tres niveis de ajuste
hr_bruto      = 0.76; ic_bruto      = (0.70, 0.83)
hr_ajust_clin = 0.85; ic_ajust_clin = (0.82, 0.89)
hr_ajust_depr = 0.91; ic_ajust_depr = (0.88, 0.94)

# Alimujiang et al., 2019, JAMA Network Open - Health and Retirement Study
# ATENCAO: contraste de EXTREMOS (menor proposito vs maior), nao por unidade
n_hrs         = 6_985
p_mulheres    = 0.575
idade_media   = 68.6
idade_dp      = 9.8
hr_hrs        = 2.43; ic_hrs        = (1.57, 3.75)
hr_hrs_cardio = 2.66; ic_hrs_cardio = (1.62, 4.38)

# Cohen, Bavishi & Rozanski, 2016 - registro historico
n_cohen       = 136_265
k_cohen       = 10

aplica_fator_080 = False

print("=" * 68)
print("  DADOS DA LITERATURA - PROPOSITO E MORTALIDADE")
print("=" * 68)
print(f"\n  Sutin et al., 2026 (Psychological Medicine):")
print(f"  -> {k_amostras} amostras · N={n_meta:,} · {obitos_meta:,} obitos"
      f" · ate {seguimento} anos")
print(f"  -> HR bruto                       {hr_bruto:.2f}  IC 95% {ic_bruto}")
print(f"  -> HR ajustado (comport./clinico) {hr_ajust_clin:.2f}  IC 95% {ic_ajust_clin}")
print(f"  -> HR ajustado (depressao)        {hr_ajust_depr:.2f}  IC 95% {ic_ajust_depr}")
print(f"\n  Alimujiang et al., 2019 (JAMA Network Open):")
print(f"  -> N={n_hrs:,} · {p_mulheres*100:.1f}% mulheres"
      f" · idade media {idade_media} (DP {idade_dp})")
print(f"  -> HR todas as causas             {hr_hrs:.2f}  IC 95% {ic_hrs}")
print(f"  -> HR cardio/circulatorio/sangue  {hr_hrs_cardio:.2f}  IC 95% {ic_hrs_cardio}")
print(f"  -> CONTRASTE DE EXTREMOS, nao por unidade da escala")
print(f"\n  Cohen, Bavishi & Rozanski, 2016: {k_cohen} estudos · N={n_cohen:,}")
print(f"\n  Fator x0.80 aplicado: {aplica_fator_080}")
print("=" * 68)

In [ ]:
# --- O MODELO ---
# Assinatura estatistica: a distribuicao do log do hazard ratio.
# De um IC 95% publicado sai o erro padrao exato, sem inventar nada:
#     ln(HR) ~ Normal( ln(HR_pontual), EP )
#     EP = [ ln(limite superior) - ln(limite inferior) ] / (2 x 1,96)

Z = stats.norm.ppf(0.975)   # 1,959964


def ep_do_ic(ic):
    return (np.log(ic[1]) - np.log(ic[0])) / (2 * Z)


ep_bruto      = ep_do_ic(ic_bruto)
ep_ajust_clin = ep_do_ic(ic_ajust_clin)
ep_ajust_depr = ep_do_ic(ic_ajust_depr)

# Reducao percentual de risco instantaneo e o inverso publicado pelos autores
reducao_bruto = (1 - hr_bruto) * 100
inverso       = 1 / hr_bruto

# Quanto da associacao bruta sobra depois de cada camada de ajuste,
# medida na escala em que o HR e aditivo: o log.
sobra_clin = np.log(hr_ajust_clin) / np.log(hr_bruto)
sobra_depr = np.log(hr_ajust_depr) / np.log(hr_bruto)

# Alimujiang na mesma direcao das demais (invertido), so para leitura visual.
# NAO e comparavel ponto a ponto: contraste de extremos contra por unidade.
hr_hrs_inv = 1 / hr_hrs
ic_hrs_inv = (1 / ic_hrs[1], 1 / ic_hrs[0])

print("=" * 68)
print("  MODELO - O EFEITO E A ATENUACAO")
print("=" * 68)
print(f"\n  Efeito bruto (N={n_meta:,}):")
print(f"  -> HR = {hr_bruto:.2f}  IC 95% [{ic_bruto[0]:.2f} · {ic_bruto[1]:.2f}]")
print(f"  -> Taxa instantanea {reducao_bruto:.0f}% menor")
print(f"  -> Lido ao contrario: 1/{hr_bruto:.2f} = {inverso:.2f}x")
print(f"  -> EP do log(HR) = {ep_bruto:.4f}")
print(f"\n  O que sobra depois do ajuste (escala log, onde o HR e aditivo):")
print(f"  -> Ajuste comportamental e clinico: sobra {sobra_clin*100:.0f}% da associacao")
print(f"  -> Ajuste por depressao:            sobra {sobra_depr*100:.0f}% da associacao")
print(f"  -> Nos tres niveis o IC 95% NAO cruza 1. O efeito encolhe e nao some.")
print(f"\n  Coorte isolada (Alimujiang, N={n_hrs:,}), contraste de extremos:")
print(f"  -> HR = {hr_hrs:.2f}  IC 95% [{ic_hrs[0]:.2f} · {ic_hrs[1]:.2f}]")
print(f"  -> Invertido para a mesma direcao: {hr_hrs_inv:.2f}"
      f"  IC 95% [{ic_hrs_inv[0]:.2f} · {ic_hrs_inv[1]:.2f}]")
print(f"  -> NAO comparar ponto a ponto com o HR por unidade")
print("=" * 68)

In [ ]:
# --- VISUALIZACAO ---

def br(n):
    return f"{n:,}".replace(",", ".")


DOURADO = '#c8a84b'
VERMELHO = '#c0392b'
VERDE = '#2a8a82'
CINZA = '#6b6a64'

# GRAFICO 1 - A atenuacao camada por camada
fig1, ax1 = plt.subplots(figsize=(12, 8))

rotulos = ['Associação bruta',
           'Ajustada por fatores\ncomportamentais e clínicos',
           'Ajustada também\npor depressão']
hrs = [hr_bruto, hr_ajust_clin, hr_ajust_depr]
ics = [ic_bruto, ic_ajust_clin, ic_ajust_depr]
cores = [VERDE, DOURADO, VERMELHO]
y = np.arange(len(hrs))[::-1]

for yi, hr, ic, cor in zip(y, hrs, ics, cores):
    ax1.plot([ic[0], ic[1]], [yi, yi], color=cor, linewidth=3, solid_capstyle='round')
    ax1.plot([ic[0], ic[0]], [yi - 0.08, yi + 0.08], color=cor, linewidth=3)
    ax1.plot([ic[1], ic[1]], [yi - 0.08, yi + 0.08], color=cor, linewidth=3)
    ax1.scatter([hr], [yi], s=260, color=cor, zorder=3)
    ax1.text(hr, yi + 0.19, f'{hr:.2f}'.replace('.', ','),
             ha='center', fontsize=17, fontweight='bold', color=cor)
    ax1.text(ic[1] * 1.012, yi, f'[{ic[0]:.2f} · {ic[1]:.2f}]'.replace('.', ','),
             va='center', fontsize=11, color=CINZA)

ax1.axvline(x=1.0, color='#333', linestyle='--', linewidth=1.5)
ax1.text(1.002, y.max() + 0.42, 'HR = 1: nenhuma associação', fontsize=11, color='#333')

ax1.set_yticks(y)
ax1.set_yticklabels(rotulos, fontsize=12)
ax1.set_xlim(0.66, 1.03)
ax1.set_ylim(-0.55, y.max() + 0.62)
ax1.set_xlabel('Hazard ratio de mortalidade por todas as causas (escala log)')
ax1.set_xscale('log')
ax1.set_xticks([0.7, 0.76, 0.85, 0.91, 1.0])
ax1.get_xaxis().set_major_formatter(plt.FuncFormatter(
    lambda v, _: f'{v:.2f}'.replace('.', ',')))
ax1.get_xaxis().set_minor_formatter(plt.NullFormatter())
ax1.tick_params(axis='x', which='minor', length=0)
ax1.set_title('O efeito encolhe a cada camada de ajuste, e não desaparece\n'
              f'Sutin et al., 2026 · {k_amostras} amostras · N={br(n_meta)} · '
              f'{br(obitos_meta)} óbitos',
              fontsize=14, pad=18)

plt.figtext(0.5, 0.005,
            'Fonte: Sutin et al., 2026, Psychological Medicine  |  #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-064-grafico-01-atenuacao.png', dpi=150, bbox_inches='tight')
plt.close()
print("Grafico 1 salvo")

# GRAFICO 2 - A coorte famosa contra a base gigante
fig2, ax2 = plt.subplots(figsize=(12, 8))

bases = [f'Coorte isolada\nAlimujiang 2019\nN={br(n_hrs)}',
         f'Meta-análise\nSutin 2026\nN={br(n_meta)}']
alturas = [n_hrs, n_meta]
barras = ax2.barh([0, 1], alturas, color=[VERMELHO, VERDE], alpha=0.85, height=0.45)
ax2.set_yticks([0, 1])
ax2.set_yticklabels(bases, fontsize=12)
ax2.set_xscale('log')
ax2.get_xaxis().set_minor_formatter(plt.NullFormatter())
ax2.set_xlim(1e3, 2e6)
ax2.set_xlabel('Participantes (escala log)')
ax2.set_title('Duas evidências, dois tamanhos, dois contrastes diferentes',
              fontsize=14, pad=18)

ax2.text(n_hrs * 1.35, 0,
         f'HR {hr_hrs:.2f}'.replace('.', ',') +
         f' [{ic_hrs[0]:.2f} · {ic_hrs[1]:.2f}]'.replace('.', ',') +
         '\ncontraste de EXTREMOS de categoria',
         va='center', fontsize=12, color=VERMELHO, fontweight='bold')
ax2.text(n_meta * 1.35, 1,
         f'HR {hr_bruto:.2f}'.replace('.', ',') +
         f' [{ic_bruto[0]:.2f} · {ic_bruto[1]:.2f}]'.replace('.', ',') +
         '\nPOR UNIDADE da escala',
         va='center', fontsize=12, color=VERDE, fontweight='bold')

ax2.text(0.5, -0.19,
         'Os dois apontam na mesma direção, mas medem contrastes diferentes.\n'
         'Não devem ser comparados ponto a ponto.',
         transform=ax2.transAxes, ha='center', fontsize=11, color=CINZA, style='italic')

plt.figtext(0.5, 0.005,
            'Fontes: Alimujiang et al., 2019, JAMA Network Open · Sutin et al., 2026, '
            'Psychological Medicine  |  #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-064-grafico-02-coorte-vs-meta.png', dpi=150, bbox_inches='tight')
plt.close()
print("Grafico 2 salvo")

# GRAFICO 3 - Assinatura estatistica: a distribuicao do log(HR)
fig3, ax3 = plt.subplots(figsize=(12, 8))

x = np.linspace(0.60, 1.05, 3000)
dist_log = stats.norm(np.log(hr_bruto), ep_bruto)
densidade = dist_log.pdf(np.log(x)) / x        # densidade na escala do HR

ax3.plot(x, densidade, color=VERDE, linewidth=3)
ax3.fill_between(x, densidade, alpha=0.18, color=VERDE)
dentro = (x >= ic_bruto[0]) & (x <= ic_bruto[1])
ax3.fill_between(x[dentro], densidade[dentro], alpha=0.30, color=VERDE)

for limite in ic_bruto:
    ax3.axvline(x=limite, color=DOURADO, linestyle='--', linewidth=2)
ax3.axvline(x=hr_bruto, color='#333', linewidth=2)
ax3.axvline(x=1.0, color=VERMELHO, linestyle=':', linewidth=2)

topo = densidade.max()
ax3.text(1.002, topo * 0.55, 'HR = 1\nnenhuma\nassociação',
         fontsize=11, color=VERMELHO)
ax3.text(hr_bruto, topo * 1.04, f'HR = 0,76', ha='center',
         fontsize=15, fontweight='bold', color='#333')
ax3.text((ic_bruto[0] + ic_bruto[1]) / 2, topo * 0.12,
         f'IC 95% [0,70 · 0,83]', ha='center', fontsize=13, color=DOURADO,
         fontweight='bold')

ax3.get_xaxis().set_major_formatter(plt.FuncFormatter(
    lambda v, _: f'{v:.1f}'.replace('.', ',')))
ax3.set_ylim(0, topo * 1.22)
ax3.set_xlabel('Hazard ratio de mortalidade por todas as causas')
ax3.set_ylabel('Densidade')
ax3.set_title('A assinatura estatística: ln(HR) ~ Normal(ln 0,76 ; EP)\n'
              f'EP = [ln({ic_bruto[1]:.2f}) − ln({ic_bruto[0]:.2f})] / (2 × 1,96) = '
              f'{ep_bruto:.4f}'.replace('.', ','),
              fontsize=14, pad=18)

ax3.text(0.5, -0.145,
         'Distribuição derivada do IC 95% publicado, sem parâmetro arbitrário',
        transform=ax3.transAxes, ha='center', fontsize=12, color=DOURADO,
         fontweight='bold')

plt.figtext(0.5, 0.005,
            'Fonte: Sutin et al., 2026, Psychological Medicine  |  #365Probabilidades',
            ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.savefig('dia-064-grafico-03-assinatura.png', dpi=150, bbox_inches='tight')
plt.close()
print("Grafico 3 salvo")

### 💡 O Insight

**488.765 pessoas. 48.928 mortes. Até 32 anos de acompanhamento.**

Quem tem mais senso de propósito morre a uma taxa cerca de **30% menor** ao longo
do seguimento. HR = 0,76, IC 95% [0,70 · 0,83].

Esse número por si só já valeria o dia. Mas o que ele faz depois é mais
interessante do que o tamanho dele.

Quando os autores ajustam por fatores comportamentais e clínicos, o HR sobe para
0,85. Quando ajustam também por depressão, sobe para 0,91.

Traduzindo: **boa parte da associação passa por caminhos que a gente já conhece.**
Quem tem propósito tende a se mexer mais, a se cuidar mais, a estar menos
deprimido. Essas coisas explicam um pedaço grande do efeito.

Mas não explicam tudo. Nos três níveis de ajuste o intervalo de confiança não
encosta em 1. O efeito encolhe e continua lá.

E existe a possibilidade que ninguém consegue eliminar em estudo observacional: a
seta pode apontar para trás. Quem já está adoecendo relata menos propósito, e
morre antes. Parte do que se vê pode ser a doença aparecendo nas duas medidas.

Então a frase honesta não é "ter um porquê faz você viver mais".

É: **ter um porquê caminha junto com viver mais, de forma consistente em três
continentes e em 32 anos de dados, e uma parte disso continua de pé mesmo depois
de descontar comportamento, doença e depressão.**

Não é uma promessa. É um sinal grande demais para ser ignorado.

*O que, hoje, faria falta se você não estivesse aqui para fazer?*

---

### ⚠️ Limitações do Modelo

- **Coorte observacional. Associação, não causa.** Ninguém sorteou propósito para
  metade da amostra, e nenhum ajuste estatístico substitui aleatorização.
- **Causalidade reversa é plausível e não eliminável.** Quem está adoecendo pode
  relatar menos propósito e morrer antes pelo próprio adoecimento.
- **Contrastes diferentes.** O HR de 2,43 do Alimujiang compara extremos de
  categoria; o HR de 0,76 é por unidade da escala. Estão no notebook lado a lado
  para leitura de direção, não para comparação ponto a ponto.
- O propósito é medido por autorrelato (escala de Ryff e medidas de item único,
  conforme o estudo). O desfecho é que é objetivo.
- Meta-análise combina 25 amostras com escalas e seguimentos diferentes. Parte da
  variação entre estudos é heterogeneidade real, não ruído.
- O HR de 3,15 do modelo menos ajustado do Alimujiang, que circulava nas minhas
  anotações, **não foi confirmado em fonte acessível e ficou fora do modelo**.
- A assinatura estatística deriva o erro padrão do IC 95% publicado. É exata sob
  normalidade do log do HR, que é a premissa padrão desse tipo de intervalo.
- Fator ×0.80 não aplicado: coorte prospectiva com desfecho objetivo, óbito
  registrado.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*
